# 03 — Modelling: BERTopic Hyperparameter Experimentation
### Topic Modelling on India News Headlines (2001–2023)

**Goal of this notebook:** run a series of BERTopic configurations — varying the embedding
model, UMAP dimensionality-reduction settings, and HDBSCAN clustering settings — score each one,
and identify the best-performing configuration with a documented justification.

**How BERTopic works, in short:**
1. **Embed** every headline into a dense vector using a sentence-Transformer (captures meaning,
   not just word overlap).
2. **Reduce dimensionality** of those embeddings with **UMAP** (Transformer embeddings are
   typically 384–768 dimensions; clustering algorithms work better, and faster, in a smaller
   space, e.g. 5–15 dimensions).
3. **Cluster** the reduced embeddings with **HDBSCAN** (density-based — finds clusters of
   varying shapes/sizes and explicitly marks low-density points as outliers/noise, rather than
   forcing every document into a cluster like k-means would).
4. **Represent each cluster as a topic** using **c-TF-IDF** — a class-based TF-IDF that finds
   the words most distinctive to each cluster compared to all others, giving each topic a
   ranked keyword list.

**Why we experiment with hyperparameters:**
- **`n_neighbors` (UMAP)** — how many nearby points UMAP considers per document. Low values
  preserve fine local structure (more, smaller topics); high values preserve broader structure
  (fewer, larger topics).
- **`n_components` (UMAP)** — how many dimensions to reduce down to. Fewer dimensions are faster
  to cluster but can lose information; more dimensions retain more structure but slow clustering.
- **`min_cluster_size` (HDBSCAN)** — the minimum number of documents to count as a valid topic.
  Low values produce many small, granular topics (some possibly noisy); high values produce
  fewer, broader topics.
- **`min_samples` (HDBSCAN)** — controls how conservative the clustering is about calling a
  point an outlier vs assigning it to a cluster. Higher values → more points marked as noise.

**Efficiency note:** embedding 285k documents through a Transformer is the expensive step.
UMAP + HDBSCAN, run on top of an *already-computed* embedding matrix, are comparatively cheap.
So this notebook computes embeddings **once per embedding model** and caches them to Drive —
the actual hyperparameter grid below only re-runs UMAP + HDBSCAN, which is what makes running
many combinations realistic.

**Run this in Google Colab with a GPU runtime** (Runtime → Change runtime type → T4 GPU) — the
embedding step is much faster with one.


In [ ]:
# Mount Drive (same persistent project folder used by notebooks 01 and 02).
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/topic-modelling-capstone'
DATA_PROCESSED = f'{BASE_DIR}/data/processed'
OUTPUTS_MODELS = f'{BASE_DIR}/outputs/models'
OUTPUTS_FIGURES = f'{BASE_DIR}/outputs/figures'
EMBEDDINGS_CACHE = f'{BASE_DIR}/outputs/embeddings'
for d in [OUTPUTS_MODELS, OUTPUTS_FIGURES, EMBEDDINGS_CACHE]:
    os.makedirs(d, exist_ok=True)


In [ ]:
# --- Install dependencies ---
# bertopic: the main modelling library (wraps embedding + UMAP + HDBSCAN + c-TF-IDF).
# sentence-transformers: the embedding models BERTopic uses.
# umap-learn / hdbscan: the dimensionality reduction and clustering steps (also BERTopic deps,
#   but we import them directly since we pass custom-configured instances into BERTopic).
# gensim: used for computing coherence scores (a standard topic-model quality metric).
!pip install -q bertopic sentence-transformers umap-learn hdbscan gensim

import pandas as pd
import numpy as np
import time
import itertools
import pickle

from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer

from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

import torch
print("GPU available:", torch.cuda.is_available())


## 1. Load preprocessed data from notebook 02

In [ ]:
df = pd.read_csv(f'{DATA_PROCESSED}/headlines_preprocessed.csv')
print(f"Loaded {len(df):,} rows")

# text_for_embedding -> fed into the sentence embedding model
# text_for_representation -> used for c-TF-IDF topic keywords and coherence scoring
docs_embed = df['text_for_embedding'].astype(str).tolist()
docs_repr = df['text_for_representation'].astype(str).tolist()

print(f"Documents ready for modelling: {len(docs_embed):,}")


## 2. Embedding models to compare, and a caching helper

We compare two embedding models with different speed/quality trade-offs:
- **`all-MiniLM-L6-v2`** — small, fast (384-dim), a strong general-purpose baseline.
- **`all-mpnet-base-v2`** — larger, slower, generally higher-quality embeddings (768-dim).

Embeddings are computed once per model and cached to Drive as `.npy` files — if you re-run this
notebook later, it loads the cached array instead of recomputing (saves a lot of time).


In [ ]:
EMBEDDING_MODELS = {
    'MiniLM-L6-v2': 'sentence-transformers/all-MiniLM-L6-v2',
    'mpnet-base-v2': 'sentence-transformers/all-mpnet-base-v2',
}

def get_embeddings(model_key, model_name, docs):
    """Return embeddings for `docs` using `model_name`, using a cached .npy file in Drive
    if one already exists, otherwise computing and caching them."""
    cache_path = f'{EMBEDDINGS_CACHE}/{model_key}_embeddings.npy'
    if os.path.exists(cache_path):
        print(f"Loading cached embeddings for {model_key} from Drive...")
        return np.load(cache_path)

    print(f"Computing embeddings for {model_key} ({model_name})... this is the slow step.")
    model = SentenceTransformer(model_name)
    start = time.time()
    embeddings = model.encode(docs, show_progress_bar=True, batch_size=256)
    elapsed = time.time() - start
    print(f"Done in {elapsed/60:.1f} minutes. Shape: {embeddings.shape}")

    np.save(cache_path, embeddings)
    print(f"Cached to {cache_path}")
    return embeddings


## 3. Compute (or load cached) embeddings for both models

This cell is the one long-running step. Once it's done, every experiment below reuses these
arrays — no re-embedding.


In [ ]:
embeddings_by_model = {}
for key, name in EMBEDDING_MODELS.items():
    embeddings_by_model[key] = get_embeddings(key, name, docs_embed)

for key, emb in embeddings_by_model.items():
    print(f"{key}: shape {emb.shape}")


## 4. Coherence & diversity evaluation helpers

**Topic coherence (c_npmi)** measures how semantically related the top words within each topic
are, based on how often they co-occur in the corpus — higher is better, and it's the standard
way to judge whether a topic's keywords "make sense together" without a human reading every one.
We use `c_npmi` (rather than `c_v`) because it's substantially faster on a large corpus while
still being a well-established coherence metric.

**Topic diversity** measures what fraction of top words across *all* topics are unique — low
diversity means topics are repeating similar keywords (a sign the model isn't distinguishing
topics well); high diversity means topics are more distinct from each other.

We evaluate on a random subsample of documents (for coherence) purely for speed — the topics
themselves are still derived from the full clustering.


In [ ]:
def compute_coherence(topic_model, docs_tokenized, top_n=10, sample_size=50_000):
    """Compute c_npmi coherence for a fitted BERTopic model."""
    # Subsample for speed — coherence computation over the full 285k-document corpus is slow
    # and the score stabilizes well before using every document.
    if len(docs_tokenized) > sample_size:
        rng = np.random.RandomState(RANDOM_STATE)
        idx = rng.choice(len(docs_tokenized), sample_size, replace=False)
        sample_docs = [docs_tokenized[i] for i in idx]
    else:
        sample_docs = docs_tokenized

    dictionary = Dictionary(sample_docs)

    topics = topic_model.get_topics()
    topic_words = [
        [word for word, _ in words[:top_n]]
        for topic_id, words in topics.items()
        if topic_id != -1  # exclude the outlier "topic"
    ]
    # Drop any topic whose words aren't in the coherence dictionary (can happen with very
    # small/rare topics on the subsample) to avoid errors.
    topic_words = [t for t in topic_words if all(dictionary.token2id.get(w) is not None for w in t)]

    if len(topic_words) < 2:
        return np.nan  # not enough valid topics to score

    cm = CoherenceModel(topics=topic_words, texts=sample_docs, dictionary=dictionary,
                         coherence='c_npmi')
    return cm.get_coherence()


def compute_diversity(topic_model, top_n=10):
    """Fraction of unique words across all topics' top-N keyword lists."""
    topics = topic_model.get_topics()
    all_words = []
    for topic_id, words in topics.items():
        if topic_id == -1:
            continue
        all_words.extend([w for w, _ in words[:top_n]])
    if len(all_words) == 0:
        return np.nan
    return len(set(all_words)) / len(all_words)


## 5. The experiment grid

Each row is one full configuration to try. Starting with **10 combinations** — comfortably
covers a first pass. To go further (20-30+), just add more dicts to this list: e.g. more
`min_cluster_size` values, the second embedding model, or additional `n_neighbors` values.
We kept the starting grid to one embedding model (the faster one) to get through a first
comparison quickly; the `mpnet-base-v2` embeddings are already cached from step 3, so adding
runs that use it later costs no extra embedding time.


In [ ]:
EXPERIMENT_GRID = [
    # --- MiniLM-L6-v2, varying UMAP n_neighbors (topic granularity) ---
    {'embedding_model': 'MiniLM-L6-v2', 'n_neighbors': 10, 'n_components': 5,  'min_cluster_size': 50,  'min_samples': None},
    {'embedding_model': 'MiniLM-L6-v2', 'n_neighbors': 15, 'n_components': 5,  'min_cluster_size': 50,  'min_samples': None},
    {'embedding_model': 'MiniLM-L6-v2', 'n_neighbors': 30, 'n_components': 5,  'min_cluster_size': 50,  'min_samples': None},

    # --- MiniLM-L6-v2, varying HDBSCAN min_cluster_size (topic count / size) ---
    {'embedding_model': 'MiniLM-L6-v2', 'n_neighbors': 15, 'n_components': 5,  'min_cluster_size': 20,  'min_samples': None},
    {'embedding_model': 'MiniLM-L6-v2', 'n_neighbors': 15, 'n_components': 5,  'min_cluster_size': 100, 'min_samples': None},
    {'embedding_model': 'MiniLM-L6-v2', 'n_neighbors': 15, 'n_components': 5,  'min_cluster_size': 200, 'min_samples': None},

    # --- MiniLM-L6-v2, varying UMAP n_components (info retained before clustering) ---
    {'embedding_model': 'MiniLM-L6-v2', 'n_neighbors': 15, 'n_components': 10, 'min_cluster_size': 50,  'min_samples': None},
    {'embedding_model': 'MiniLM-L6-v2', 'n_neighbors': 15, 'n_components': 15, 'min_cluster_size': 50,  'min_samples': None},

    # --- MiniLM-L6-v2, varying HDBSCAN min_samples (outlier strictness) ---
    {'embedding_model': 'MiniLM-L6-v2', 'n_neighbors': 15, 'n_components': 5,  'min_cluster_size': 50,  'min_samples': 10},
    {'embedding_model': 'MiniLM-L6-v2', 'n_neighbors': 15, 'n_components': 5,  'min_cluster_size': 50,  'min_samples': 25},
]

print(f"{len(EXPERIMENT_GRID)} configurations queued.")
print("To expand to 20-30+: duplicate rows above with 'embedding_model': 'mpnet-base-v2', "
      "or add more values for any parameter.")


## 6. Run the grid

For each configuration: build UMAP + HDBSCAN with those settings, fit BERTopic on the
corresponding cached embeddings, score it, and log the result. This is the main time-consuming
cell (though far cheaper than re-embedding each time) — expect each run to take a few minutes.


In [ ]:
# Tokenize the representation text once, upfront — reused for every coherence computation.
docs_tokenized = [d.split() for d in docs_repr]

# c-TF-IDF and vectorizer settings shared across all runs — kept constant so the comparison
# isolates the effect of the UMAP/HDBSCAN parameters being varied, not the keyword extraction step.
vectorizer_model = CountVectorizer(stop_words='english', min_df=5, ngram_range=(1, 2))
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

results = []
fitted_models = {}  # keep fitted models in memory for the best one(s) to inspect later

for i, cfg in enumerate(EXPERIMENT_GRID):
    run_id = f"run_{i:02d}"
    print(f"\n--- {run_id}: {cfg} ---")
    start = time.time()

    embeddings = embeddings_by_model[cfg['embedding_model']]

    umap_model = UMAP(
        n_neighbors=cfg['n_neighbors'],
        n_components=cfg['n_components'],
        min_dist=0.0,          # standard for clustering use-cases (vs visualization)
        metric='cosine',
        random_state=RANDOM_STATE,
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=cfg['min_cluster_size'],
        min_samples=cfg['min_samples'],
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True,
    )

    topic_model = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        ctfidf_model=ctfidf_model,
        calculate_probabilities=False,
        verbose=False,
    )

    topics, _ = topic_model.fit_transform(docs_repr, embeddings=embeddings)

    n_topics = len(set(topics)) - (1 if -1 in topics else 0)
    n_outliers = sum(1 for t in topics if t == -1)
    pct_outliers = n_outliers / len(topics)

    coherence = compute_coherence(topic_model, docs_tokenized)
    diversity = compute_diversity(topic_model)

    elapsed = time.time() - start

    results.append({
        'run_id': run_id,
        **cfg,
        'n_topics': n_topics,
        'pct_outliers': round(pct_outliers, 4),
        'coherence_npmi': round(coherence, 4) if not np.isnan(coherence) else np.nan,
        'diversity': round(diversity, 4) if not np.isnan(diversity) else np.nan,
        'runtime_sec': round(elapsed, 1),
    })
    fitted_models[run_id] = topic_model

    print(f"  -> {n_topics} topics | {pct_outliers:.1%} outliers | "
          f"coherence={coherence:.4f} | diversity={diversity:.4f} | {elapsed:.0f}s")

results_df = pd.DataFrame(results)
results_df


## 7. Compare results

We rank primarily by **coherence** (are the topics semantically sensible?), using **diversity**
as a tie-breaker (are topics distinct from each other, not redundant?), while also sanity
checking **`n_topics`** (too few is uninformative; too many is over-fragmented) and
**`pct_outliers`** (very high values mean the model is failing to cluster most documents at all).


In [ ]:
results_df_sorted = results_df.sort_values('coherence_npmi', ascending=False)
results_df_sorted


In [ ]:
results_df.to_csv(f'{OUTPUTS_MODELS}/experiment_results.csv', index=False)
print(f"Saved experiment log to {OUTPUTS_MODELS}/experiment_results.csv")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=results_df, x='n_topics', y='coherence_npmi', hue='min_cluster_size',
                 palette='viridis', s=100, ax=axes[0])
axes[0].set_title('Coherence vs number of topics found')

sns.scatterplot(data=results_df, x='pct_outliers', y='coherence_npmi', hue='embedding_model',
                 s=100, ax=axes[1])
axes[1].set_title('Coherence vs outlier percentage')

plt.tight_layout()
plt.savefig(f'{OUTPUTS_FIGURES}/hyperparameter_comparison.png', dpi=150)
plt.show()


## 8. Select and save the best model

*(Once you've reviewed `results_df_sorted`, fill in which run won and why — this becomes the
"Best Model and why" section of the report.)*


In [ ]:
BEST_RUN_ID = results_df_sorted.iloc[0]['run_id']  # highest coherence by default — override
                                                     # this string if you decide a different run
                                                     # is the better pick after reviewing tradeoffs
print(f"Selected best run: {BEST_RUN_ID}")
print(results_df_sorted[results_df_sorted['run_id'] == BEST_RUN_ID])

best_model = fitted_models[BEST_RUN_ID]

# Save the fitted BERTopic model to Drive so notebook 04 (evaluation/results) can load it
# directly without refitting.
best_model.save(f'{OUTPUTS_MODELS}/best_bertopic_model', serialization='pickle')
print(f"Saved best model to {OUTPUTS_MODELS}/best_bertopic_model")


In [ ]:
# Preview the topics the best model found
best_model.get_topic_info().head(20)


## 9. Summary of modelling experiments

*(Fill in with actual numbers/observations once you've run the grid — this maps directly onto
the report's "Modelling" and "Best Model and why" sections.)*

- Number of configurations tried: `<fill in>` (embedding model(s): `<fill in>`)
- Best run: `<run_id>` — `<parameter values>`
- Best run's metrics: `<n_topics>` topics, `<pct_outliers>` outliers, coherence `<value>`,
  diversity `<value>`
- Observed pattern for `n_neighbors`: `<e.g. lower values fragmented topics too much / higher
  values merged distinct topics together>`
- Observed pattern for `min_cluster_size`: `<fill in>`
- Observed pattern for embedding model choice (if both tried): `<fill in>`
- Why this configuration was selected as best: `<tie the metrics back to what makes a topic
  model genuinely useful here — sensible, distinct, well-populated topics with a manageable
  outlier rate>`
